# define

## TODO:
- izlabot bug, kad contains() ņem ne tikai MRK1.1 bet arī MRK1.10 MRK 1.11 utt
- unificēt atslēgu nosaukumus lielo 4 dzīvo būtņu (objektu) vārdnīcās
- salikt flagus, lai redzami vārdi kuriem vēl netika atrasts atbilstošs (vēlāk noderēs salīdzinot atvasinājumus)
- parādīt purpura krāsā ja pārbīdīts vārds, un par cik
- sāk meklēšanu kā tagad tieši no 0 ofseta, tad nākamo skatās +- pēdējais lietotias ofsets. ja nav, tad bīdās no o pa biškum un no ofseta pa biškum.


In [ ]:
from pathlib import Path
from bs4 import BeautifulSoup
import os
from typing import List, Dict, Tuple
import json
from datetime import datetime
import re
from lxml import etree
import json

import pandas as pd

nt_stubchapters = ["matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]

def element_to_dict(element):
    """Convert an lxml Element to a dictionary that can be JSON serialized"""
    result = {}
    
    if element.attrib:
        result['@attributes'] = dict(element.attrib)
    
    if element.text and element.text.strip():
        result['#text'] = element.text.strip()
    
    children = {}
    for child in element:
        if len(child) == 0:
            if child.text and child.text.strip():
                children[child.tag] = child.text.strip()
        else:
            children[child.tag] = element_to_dict(child)
    
    if children:
        result.update(children)
    
    return result

def extract_text_and_number(text):
    pattern = r'^([a-zA-Z\s\d]+)\s+(\d+)$'
    match = re.match(pattern, text.strip())
    return (match.group(1).strip().lower(), int(match.group(2))) if match else None


def tt(ch: str):
    return tr_ch_el_bh(ch)
def tr_ch_el_bh(ch: str):
    return ch.replace(" ", "_").lower()


# define parsers

In [ ]:

class ElielTreeBankXMLParser:
    def __init__(self):
        self.xml_parser = etree.XMLParser(resolve_entities=False, no_network=True)
        
    def parse_xml_source_proiel(self, xml_path: str, save_result= False) -> Dict[str, List[Dict]]:
        """Parse XML source from PROIEL project"""
        parser = self.xml_parser
        tree = etree.parse(xml_path, parser=parser)
        root = tree.getroot()
        
        result = {}
        bad_result = {}
        empty_result = {}
        df_result = []
        current_book = None
        
        # Use lxml's faster XPath expressions
        divs = root.xpath(".//div")
        ####attrdict = {}
        
        chdone = 0
        totchs = len(divs)
        for d in divs:
            current_book, chapter = extract_text_and_number(d.find("title").text.lower())
            current_book = tt(current_book)
            result[f"{current_book}_{chapter}"] = []
            print(f"{current_book} {chapter} ( {float(chdone)/float(totchs)*100:.1f}% )")
            #print(current_book == "hebrews")
            
            verse_idx = 1
            sentences = d.xpath('sentence')
            for stn in range(len(sentences)):
                sentence = sentences[stn]
                #sentence_tokens = []
                word_idx = 0
                
                tokens = sentence.xpath('token')
                for ct in range(len(tokens)):
                    token = tokens[ct]
                    cittpartemtytokenswtf = token.attrib.get('citation-part', '')
                    
                    if not cittpartemtytokenswtf:
                        if f"{current_book}_{chapter}" not in bad_result:
                            bad_result[f"{current_book}_{chapter}"] = []
                        bad_result[f"{current_book}_{chapter}"].append(element_to_dict(token))
                        continue
                    emptytokenmarker = token.attrib.get('empty-token-sort', '')
                    if emptytokenmarker:
                        if f"{current_book}_{chapter}" not in empty_result:
                            empty_result[f"{current_book}_{chapter}"] = []
                        empty_result[f"{current_book}_{chapter}"].append(element_to_dict(token))
                        continue
                    #####for attr in token.attrib:
                        ####attrdict[attr] = f"token.attrib.get('{attr}', ''),"
                    versenum = int(token.attrib['citation-part'].split(".")[-1])
                    if(len(result[f"{current_book}_{chapter}"])<versenum):
                        for jj in range(len(result[f"{current_book}_{chapter}"]),versenum):
                            result[f"{current_book}_{chapter}"].append([])

                    xdata = {
                        ######## text is inside form attribute(!)                        'text': token.text.strip(),
                        'verse': versenum,
                        'word': word_idx,
                        'id' : token.attrib.get('id', ''),
                        'form' : token.attrib.get('form', ''),
                        'citation-part' : token.attrib.get('citation-part', ''),
                        'lemma' : token.attrib.get('lemma', ''),
                        'part-of-speech' : token.attrib.get('part-of-speech', ''),
                        'morphology' : token.attrib.get('morphology', ''),
                        'head-id' : token.attrib.get('head-id', ''),
                        'relation' : token.attrib.get('relation', ''),
                        'presentation-after' : token.attrib.get('presentation-after', ''),
                        'information-status' : token.attrib.get('information-status', ''),
                        'empty-token-sort' : token.attrib.get('empty-token-sort', ''),
                        'antecedent-id' : token.attrib.get('antecedent-id', ''),
                        'presentation-before' : token.attrib.get('presentation-before', ''),
                        'contrast-group' : token.attrib.get('contrast-group', ''),
                        'sentence': stn,
                        ####"attrs": dict(token.attrib)
                    }
                    #sentence_tokens.append(xdata)
                    result[f"{current_book}_{chapter}"][versenum-1].append(xdata)
                    xdata['book'] = current_book
                    xdata['chapter'] = chapter
                    df_result.append(xdata)
                    word_idx += 1
            chdone += 1
            
        if(save_result):
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            
            filename = f"{timestamp}_badparses.txt"
            with open(filename, "w") as file:
                json.dump(bad_result, file, indent=2)
            filename = f"{timestamp}_emptytokens.txt"
            with open(filename, "w") as file:
                json.dump(empty_result, file, indent=2)

            dbname = f"{timestamp}_el_proiel_el.csv"
            df = pd.DataFrame(df_result)
            Path(dbname).parent.mkdir(parents=True, exist_ok=True)
            df.to_csv(
                dbname,
                index=False,
                encoding='utf-8',
                quoting=1, 
                escapechar='\\'
            )
        return result

def str_getfrom_token_first(s1: str, tok: str):
    if not tok: return str
    pos = s1.find(tok)
    return s1[pos:] if pos >= 0 else s1

class BibleHubHTMLParser:
    def __init__(self):
        pass
    
    def parse_html_folder_biblehub(self, folder_path: str, book_filter: List[str]= None, save_result=False) -> Dict[str, List[List[str]]]:
        """Parse HTML files from BibleHub"""
        result = {}
        sentence_tokens = []
        df_result = []
        iterator_folders = []
        if book_filter is None:
            iterator_folders = os.listdir(folder_path)
        else:
            #if any(book.lower() in folder.lower() for book in book_filter)
            #dircont =
            iterator_folders = [n.lower() for n in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, n))
                       and n.lower() in book_filter]
            #for book in book_filter:
            #    if book.lower()  in dircont: iterator_folders.append(book)
        urls_greek = {}
        chdone = 0
        totchs = len(iterator_folders)
        for fold in iterator_folders:
            chapsnames = [filename for filename in os.listdir(folder_path.strip(os.path.sep)+os.path.sep+fold) 
                          if filename.endswith('.htm') or filename.endswith('.html')]
            ccchdone = 0
            totchhh = len(chapsnames)
            for filename in chapsnames:
                if(True):
                    chapter = os.path.splitext(filename)[0]
                    current_book = fold
                    result[f"{current_book}_{chapter}"] = []
                    print(f"{current_book} {chapter} ( {float(chdone)/float(totchs)*float(100)+(float(1)/float(totchs))*(float(ccchdone)/float(totchhh))*float(100):.1f}% )")
                    soup = BeautifulSoup(open(os.path.join(folder_path, fold, filename)), 'html.parser')
                    
                    table_spans = soup.find_all('table', class_='tablefloat')
                    curr_verse = 1
                    strong_num=-1
                    strong_title = ''
                    strong_en_title = ''
                    translit=''
                    translit_title=''
                    form = ''
                    form_en = ''
                    curr_node = None
                    word_idx = 0
                    for tbc in range(len(table_spans)):
                        table = table_spans[tbc]
                        #verse number?
                        curr_node = table.find("span", class_='reftop3')
                        if curr_node is not None:
                            # nbsp;
                            curr_verse = int(curr_node.text.strip("\xa0"))
                            word_idx = 0
                        
                        if(len(result[f"{current_book}_{chapter}"])<curr_verse):
                            for jj in range(len(result[f"{current_book}_{chapter}"]),curr_verse):
                                result[f"{current_book}_{chapter}"].append([])

                        #strongs title?
                        curr_node = table.find("span", class_='pos')
                        if curr_node is not None:
                            curr_node = curr_node.find('a')
                            if curr_node is not None:
                                strong_num = int(curr_node.text)
                                strong_title = curr_node.text + str_getfrom_token_first(curr_node.attrs.get('title', ''), ':')
                                attrb = curr_node.attrs.get('href', '')
                                if(attrb != ''):
                                    if(attrb.startswith('/greek')):
                                        urls_greek[attrb]=f"curl -O https://www.biblehub.com{attrb}"
                                        
                        #englishmans concordance
                        curr_node = table.find("span", class_='strongsnt2')
                        if curr_node is not None:
                            curr_node = curr_node.find('a')
                            if curr_node is not None:
                                strong_en_title = curr_node.text
                                attrb = curr_node.attrs.get('href', '')
                                if(attrb != ''):
                                    if(attrb.startswith('/greek')):
                                        urls_greek[attrb]=f"curl -O https://www.biblehub.com{attrb}"
                            
                        #translit
                        curr_node = table.find("span", class_='translit')
                        if curr_node is not None:
                            curr_node = curr_node.find('a')
                            if curr_node is not None:
                                attrb = curr_node.attrs.get('href', '')
                                if(attrb != ''):
                                    if(attrb.startswith('/greek')):
                                        urls_greek[attrb]=f"curl -O https://www.biblehub.com{attrb}"
                                translit_title = curr_node.attrs.get('title', '').replace("\xa0", ' ')
                                translit = curr_node.text.replace("\xa0", ' ')
                        
                        #greek text
                        curr_node = table.find("span", class_='greek')
                        if curr_node is not None:
                            form = curr_node.text.replace("\xa0", ' ')
                        
                        #en text
                        curr_node = table.find("span", class_='eng')
                        if curr_node is not None:
                            form_en = curr_node.text.replace("\xa0", ' ')
                                    
                        xdata = {
                        ######## text is inside form attribute(!)                        'text': token.text.strip(),
                        'verse': curr_verse,
                        'word': word_idx,
                        #'id' : token.attrib.get('id', ''),
                        'form' : form,
                        'form_en': form_en,
                        #'citation-part' : token.attrib.get('citation-part', ''),
                        #'lemma' : token.attrib.get('lemma', ''),
                        #'part-of-speech' : token.attrib.get('part-of-speech', ''),
                        #'morphology' : token.attrib.get('morphology', ''),
                        #'head-id' : token.attrib.get('head-id', ''),
                        #'relation' : token.attrib.get('relation', ''),
                        #'presentation-after' : token.attrib.get('presentation-after', ''),
                        #'information-status' : token.attrib.get('information-status', ''),
                        #'empty-token-sort' : token.attrib.get('empty-token-sort', ''),
                        #'antecedent-id' : token.attrib.get('antecedent-id', ''),
                        #'presentation-before' : token.attrib.get('presentation-before', ''),
                        #'contrast-group' : token.attrib.get('contrast-group', ''),
                        ####"attrs": dict(token.attrib)
                        'strong_num': strong_num,
                        'strong_title' : strong_title,
                        'strong_en_title' : strong_en_title,
                        'translit': translit,
                        'translit_title': translit_title
                    }
                        result[f"{current_book}_{chapter}"][curr_verse-1].append(xdata)
                        xdata['book'] = current_book
                        xdata['chapter'] = chapter
                        df_result.append(xdata)
                        word_idx += 1
                ccchdone +=1
            chdone += 1

        if(save_result):
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"{timestamp}_dload.sh"
            with open(filename, "w") as file:
                for k, v in urls_greek.items():
                    file.write(f"{v}\n")
            dbname = f"{timestamp}_biblehub_el_en_direct.csv"
                # Create DataFrame and write to CSV
            df = pd.DataFrame(df_result)
            
            # Ensure output directory exists
            Path(dbname).parent.mkdir(parents=True, exist_ok=True)
            
            # Write to CSV with proper formatting
            df.to_csv(
                dbname,
                index=False,
                encoding='utf-8',
                quoting=1, 
                escapechar='\\'
            )                    
        return result

## bible.com offlined parser

In [ ]:
import json
PATH_65 = "sr/"
PATH_24 = "sr/"
PATH_BH = "b/"
PATH_EL = "xm/"
bks = []
with open(f'{PATH_65}488.json', 'r') as f:
    jsn = json.load(f)
    book_attrib = 'usfm'
#    print(jsn['books'])
    bks = [b[book_attrib] for b in jsn['books'] if book_attrib in b]
print(bks)

jd_only=True
bkslist = ["genesis", "exodus", "leviticus", "numbers", "deuteronomy", "joshua", "judges", "ruth", "1_samuel", "2_samuel", "1_kings", "2_kings", "1_chronicles", "2_chronicles", "ezra", "nehemiah", "esther", "job", "psalms", "proverbs", "ecclesiastes", "songs", "isaiah", "jeremiah", "lamentations", "ezekiel", "daniel", "hosea", "joel", "amos", "obadiah", "jonah", "micah", "nahum", "habakkuk", "zephaniah", "haggai", "zechariah", "malachi", "matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]
chps= [int(i) for i in "50 40 27 36 34 24 21 4 31 24 22 25 29 36 10 13 10 42 150 31 12 8 66 52 5 48 12 14 3 9 1 4 7 3 3 3 2 14 4 28 16 24 21 28 16 16 13 6 6 4 4 5 3 6 4 3 1 13 5 5 3 5 1 1 1 22".split(" ")]
idxl = {}
names = {}
revnames = {}
nt_stubchapters = ["matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]
for i in range (len(bkslist)):
    if (bkslist[i] in nt_stubchapters if jd_only else True):
        #idxl[bkslist[i].replace("_", "")[:3].upper()] = chps[i]
        idxl[bks[i]] = chps[i]
        names[bks[i]] = bkslist[i]
        revnames[bkslist[i]] = bks[i]

ajd_only=True
shrts = [ k for k in idxl.keys()]
abkslist = ["genesis", "exodus", "leviticus", "numbers", "deuteronomy", "joshua", "judges", "ruth", "1_samuel", "2_samuel", "1_kings", "2_kings", "1_chronicles", "2_chronicles", "ezra", "nehemiah", "esther", "job", "psalms", "proverbs", "ecclesiastes", "songs", "isaiah", "jeremiah", "lamentations", "ezekiel", "daniel", "hosea", "joel", "amos", "obadiah", "jonah", "micah", "nahum", "habakkuk", "zephaniah", "haggai", "zechariah", "malachi", "matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]
achps= [int(i) for i in "50 40 27 36 34 24 21 4 31 24 22 25 29 36 10 13 10 42 150 31 12 8 66 52 5 48 12 14 3 9 1 4 7 3 3 3 2 14 4 28 16 24 21 28 16 16 13 6 6 4 4 5 3 6 4 3 1 13 5 5 3 5 1 1 1 22".split(" ")]
bcom_to_bhub = {}
bhub_to_bcom = {}
c = 0
ant_stubchapters = ["matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]
for i in range (len(abkslist)):
    if (abkslist[i] in ant_stubchapters if ajd_only else True):
        #idxl[bkslist[i].replace("_", "")[:3].upper()] = chps[i]
        bcom_to_bhub[shrts[c]] = abkslist[i]
        bhub_to_bcom[abkslist[i]] = shrts[c]
        for jjj in range(1, idxl[shrts[c]]+1):
            bhub_to_bcom[f"{abkslist[i]}_{jjj}"] = f"{shrts[c]}_{jjj}"
        c+=1

def bh_bc(s):
    return bhub_to_bcom[s]


estr = ""
sepend ="; "    
sep =" "
for kp in idxl: estr += f"{kp}{sep}{idxl[kp]}{sepend}"
print(estr)
estr = ""
for kp in names: estr += f"{kp}{sep}{names[kp]}{sepend}"
print(estr)
def rnm(nm):
    return revnames[nm] if nm in revnames else nm

In [ ]:
#import bs4
from bs4 import BeautifulSoup

CONF_REMOVE_VERSE_NUMS = True
GREEN = '\033[92m'
RED = '\033[91m'
DIFF_BLUE =  '\x1b[38;5;12m\x1b[48;15;7m'
RESET = '\033[0m'
result ={}
result24 ={}

def html_verse_get_text_array(verses):
    result = []
    for v in verses:
        result.extend(html_verse_get_text(v))
    return result

def startswithInclasses(l, what):
    for c in l:
        if(c.startswith(what)): return True
    return False

def versetextfilter(elt):
    #print(elt.attrs['class'])
    return (elt.name == 'span' and 
            elt.has_attr('class') and 
            startswithInclasses(elt.attrs['class'], 'ChapterContent_content') and 
            elt.text.strip() != '')

def html_verse_get_text(v):
    contentels =v.find_all(versetextfilter)
    text = ""
    for v in contentels:
        if(text !=""): text += " "
        #if(CONF_REMOVE_VERSE_NUMS):
        text += v.get_text().lstrip('0123456789')\
                                       .replace(',', '').replace('.', '').replace('!', '')\
                                       .replace('?', '').replace(':', '').replace('"', '')\
                                       .replace('\'', '')
    return [{"form": i} for i in text.split()] 
        

for current_book in idxl:
    for chapter in range(1, idxl[current_book]+1):
        with open(f"{PATH_24}24_{current_book}_{chapter}.html", 'r') as f24:
            with open(f"{PATH_65}/65_{current_book}_{chapter}.html", 'r') as f:
                result[f"{current_book}_{chapter}"] = []
                result24[f"{current_book}_{chapter}"] = []

                cntnt = f.read()
                cntnt24 = f24.read()
                soup = BeautifulSoup(cntnt, 'html.parser')
                # startswith -> equals
                verses = soup.find_all('span', 
                      attrs={'data-usfm': lambda x: x and x.startswith(f"{current_book}.{chapter}")})
                ##NOTOK  filter(verses, key=attrib={'data-usfm': lambda x: x and x.startswith(f"{current_book}.{chapter}.{i}")})
                #filter(lambda x: x.attrib.get('data-usfm', '').startswith(f"{current_book}.{chapter}.{i}"), verses)
                num = max([int(e.attrs["data-usfm"].split(".")[-1]) for e in verses])
                #num = max(verses, key=lambda e: int(e.attrib["data-usfm"].split(".")[-1])).attrib["data-usfm"].split(".")[-1]
                result[f"{current_book}_{chapter}"] = [
                    html_verse_get_text_array(filter(lambda x: x.attrs.get('data-usfm', '') == (f"{current_book}.{chapter}.{i}"), verses)) for i in range(1, num+1)
                ]


                soup24 = BeautifulSoup(cntnt24, 'html.parser')
                
                verses24 = soup24.find_all('span', 
                      attrs={'data-usfm': lambda x: x and x.startswith(f"{current_book}.{chapter}")})
                num24 = max([int(e.attrs["data-usfm"].split(".")[-1]) for e in verses24])
                result24[f"{current_book}_{chapter}"] = [
                    html_verse_get_text_array(filter(lambda x: x.attrs.get('data-usfm', '') == (f"{current_book}.{chapter}.{i}"), verses24)) for i in range(1, num24+1)
                ]
                COL = f"{RED}" if len(verses) != len(verses24) else f"{GREEN}"
                print(f"{COL}{current_book}_{chapter}{RESET}")
                if(len(verses) != len(verses24)):
                    print(f"{COL}{len(verses)}'\t'{len(verses24)}{RESET}")
                #assert(verses==25)
                #print(result)
                #break
            #break
#print(result)
#soup.get_text()
print("all fulfilled")
#for v in verses:
#    result[f"{current_book}_{chapter}"].append(sentence_tokens)
#print(len(verses))
#assert(verses==25)

In [ ]:
#!mamba install -y lxml
#!mamba install -y pandas

# start work

In [ ]:
parser_html = BibleHubHTMLParser()
#html_sources = comparator.parser_html.parse_html_folder_biblehub('../../00rebordn2/', nt_stubchapters, True)
#html_sources = comparator.parser_html.parse_html_folder_biblehub('../../00rebordn2/', ["hebrews"], False)
html_sources = parser_html.parse_html_folder_biblehub('b/', nt_stubchapters, True)

In [ ]:
parser_xml = ElielTreeBankXMLParser()
##xml_sources = comparator.parser_xml.parse_xml_source_proiel('./greek-nt.xml', True)#True)
xml_sources = parser_xml.parse_xml_source_proiel('./xm/greek-nt.xml', True)#True)
#xml_sources = comparator.parser_xml.parse_xml_source_proiel('./greek-nt-syn.xml', True)#True)
def tt(ch: str):
    return tr_ch_el_bh(ch)
def tr_ch_el_bh(ch: str):
    return ch.replace(" ", "_").lower()

def ta(ch: str):
    return tr_ch_bh_el(ch)
def tr_ch_bh_el(ch: str):
    #return ch.replace("_", " ").lower()
    return ch[0].upper()+ ch[1:]

def ta(c: str):
    return c


# check data compatibility

In [ ]:
print(xml_sources[ta('hebrews_9')][26][1])
print(html_sources[('hebrews_9')][26][1])
print(result[bhub_to_bcom['hebrews_9']][26][1])
print(result24[bh_bc('hebrews_9')][26][1])

In [ ]:
print(result.keys())

# check hebs

In [ ]:
print(xml_sources[ta('hebrews_9')][26][1])
print(html_sources[('hebrews_9')][26][1])

samecnt = 0
totalcnt = 0
diffcnt = 0
diffids =[]
for iii in range(1, 14):
    for jjj in range (len(xml_sources[ta('hebrews_'+str(iii))])):
        for kkk in range (len(xml_sources[ta('hebrews_'+str(iii))][jjj])):
            totalcnt+=1
            x = xml_sources[ta('hebrews_'+str(iii))][jjj][kkk]
            print('hebrews_'+str(iii)+'::'+str(jjj)+'->'+str(kkk))
            y = html_sources[ta('hebrews_'+str(iii))][jjj][kkk]
            if(x == y):
                samecnt+=1
            else:
                diffcnt +=1
                diffids.append(((iii, jjj, kkk), (x, y)))
print(f"total: {totalcnt} same: {samecnt} diff: {diffcnt}")
print(f"total: {totalcnt} same: {float(samecnt)/float(totalcnt)*float(100):.1f}% diff: {float(diffcnt)/float(totalcnt)*float(100):.1f}%")
    

# visul rende

In [ ]:
#mport copy
xm_flags ={}
hm_flags = {k: [] for k in html_sources.keys()}

for k in xml_sources.keys():
    xm_flags[k] = []
    for c in range(len(xml_sources[k])):
        for v in range(len(xml_sources[k][c])):
            xm_flags[k].append([])
            for w in range(len(xml_sources[k][c][v])):
                xm_flags[k][v].append([])
                xm_flags[k][v][w] = False

for k in html_sources.keys():
    hm_flags[k] = []
    for c in range(len(html_sources[k])):
        for v in range(len(html_sources[k][c])):
            hm_flags[k].append([])
            for w in range(len(html_sources[k][c][v])):
                hm_flags[k][v].append([])
                hm_flags[k][v][w] = False                

print(xm_flags.keys())
print(hm_flags.keys())


In [ ]:
xm_flags["hebrews_9"][1][1]

In [ ]:
def ta(s):
    return s
#print(f"x: {xml_sources[ta("hebrews_1")][0][0]}")
#print(f"h: {html_sources["hebrews_1"][0][0]}")
# ANSI color codes
GREEN = '\033[92m'
RED = '\033[91m'
#ESC[38:5:⟨n⟩m Select foreground color      where n is a number from the table below
#ESC[48:5:⟨n⟩m Select background color
RED_W =  '\x1b[38;5;88m\x1b[48;5;7m'
DIFF_BLUE =  '\x1b[38;5;12m\x1b[48;15;7m'
DIFF_RED =  '\x1b[38;5;196m\x1b[48;15;7m'
#default bg, dark red letter
#'\x1b[38;5;88m'
#blue light bg, white letter
#'\x1b[5;46;88m'
#green bg whitel etter
#'\x1b[5;46;42m'
#green bg oragneleter'\x1b[5;31;42m'
# #'\033[38;5;55'
YELLOW = '\033[33m'
RESET = '\033[0m'
COLOR = RESET

class Visualizer_Cosole():
    def __init__(self):
        self.similars ={
    "θ" : "Θ",
    "Θ" : "θ"
}
        pass
    
    def letters_same(self, l1, l2):
        return self.words_same(l1, l2)
    
    def letters_similiar(self, l1, l2):
        if(l1.lower() == l2.lower()): return True
        sim = self.similars.get(l1.lower(), '').lower()
        if(sim!= ''): return sim == l2.lower()
        sim = self.similars.get(l2.lower(), '').lower()
        if(sim!= ''): return sim == l1.lower()
        return False
    
    def words_same(self, w1, w2):
        return w1 ==w2
    
    def getdiff_word_Visual(self, v, w, xx: str, yh: str, showGreens: bool = False, showYellows: bool = False)-> tuple[bool, bool, str]:
        p1 =""
        hadRed = False
        allReds = True
        COLOR = ''
        if(self.words_same(xx,yh)):
            allReds = False
            if not showGreens: return (hadRed, allReds, '')
            #COLOR = GREEN
            #s1 = f"{xx}"
            #s2 = f"{yh}{RESET}"
            return (hadRed, allReds, self.w_colordiff_OK(v, w, xx, yh))
        else:
            
            minlen = min(len(xx), len(yh))
            s1=""
            s2=""
            for i in range(minlen):
                if(xx[i]==yh[i]):
                    allReds = False
                    s1+=f"{GREEN}{xx[i]}{RESET}"
                    s2+=f"{GREEN}{yh[i]}{RESET}"
                    continue
                else:
                    if(self.letters_similiar(xx[i], yh[i])):
                        allReds = False
                        s1+=f"{YELLOW}{xx[i]}{RESET}"
                        s2+=f"{YELLOW}{yh[i]}{RESET}"
                        continue
                    hadRed = True
                    s1+=f"{RED}{xx[i]}{RESET}"
                    s2+=f"{RED}{yh[i]}{RESET}"
            #print(f"minlen: {minlen}")
            if(len(xx)> minlen):
                s1+=f"{RED}{xx[minlen:]}{RESET}"
            if(len(yh)> minlen):
                s2+=f"{RED}{yh[minlen:]}{RESET}"
            if(True):
                if(hadRed):
                    COLOR = RED
                else:
                    if not showYellows: return (hadRed, allReds, '')
                    COLOR = YELLOW
        p1 = f"{COLOR}{v}:{w}{RESET}"
        return (hadRed, allReds, f"{p1}\n{s1}\n{s2}")
    
    def w_colordiff_OK(self, v, w, xx: str, yh: str)-> str:
        COLOR = GREEN
        s1 = f"{xx}"
        s2 = f"{yh}{RESET}"
        p1 = f"{COLOR}{v}:{w}{RESET}"
        return f"{p1}\n{s1}\n{s2}"
            


seperator= ' '
dw = Visualizer_Cosole()
book = "hebrews"
ch_nr = 1
chapt = book+"_"+str(ch_nr)
#v = 0
xm_c_len = len(xml_sources[ta(chapt)])
hm_c_len = len(html_sources[chapt])

minverses = min(xm_c_len, hm_c_len)
for v in range(minverses):
    w = 0
    xm_v_len = len(xml_sources[ta(chapt)][v])
    hm_v_len = len(html_sources[chapt][v])
    print(f"~~~~~~b:___ {book[:3].upper()} {ch_nr}:{v+1} ___:d~~~~~~")
    shorter_x = False
    if(xm_v_len!= hm_v_len):
        if(xm_v_len<hm_v_len): shorter_x = True
        print(f"{RED_W}xml: {xm_v_len} w\nhml: {hm_v_len} w ====== ({float(xm_v_len-hm_v_len)/float(xm_v_len+hm_v_len)*float(100):.1f}% )\n{RESET}~~~~~")
    else:
        print(f"{GREEN}wrd: {xm_v_len}\n{RESET}~~~~~")
    minwords = min (xm_v_len, hm_v_len)
    for i in range (minwords):
        w = i
        xx = xml_sources[ta(chapt)][v][w]["form"]
        yh = html_sources[chapt][v][w]["form"]
        hR, aR, resultGUI = dw.getdiff_word_Visual(v+1, w+1, xx, yh, showYellows=True)
        if(resultGUI):
            print(resultGUI)
    if(shorter_x):
        strbuild =''
        for i in range(xm_v_len, hm_v_len):
            strbuild += f"{html_sources[chapt][v][i]['form']}{seperator}"
        print(f"{DIFF_RED}+{strbuild}{RESET}")
    else : 
        if (hm_v_len < xm_v_len):
            strbuild =''
            
            for i in range(hm_v_len, xm_v_len):
                strbuild += f"{xml_sources[ta(chapt)][v][i]['form']}{seperator}"
            print(f"{DIFF_BLUE}-{strbuild}{RESET}")
# write a function to visualize the differences in the console                      
            
        


# te html gennnnessiis :))))

In [ ]:
class Visualizer_HTML:
    def __init__(self):
        self.similars = {
            "θ" : "Θ",
            "Θ" : "θ"
        }
        self.html = []
        self.html.append("""
        <!DOCTYPE html>
        <html>
        <head>
            <style>
                .diff-table { border-collapse: collapse; width: 100%; }
                .diff-row { height: 1.5em; }
                .common { background-color: #e6ffe6; }
                .diff { background-color: #ffe6e6; }
                .similar { background-color: #fff7e6; }
                .header { background-color: #f0f0f0; }
            </style>
        </head>
        <body>
        """)

    def letters_similiar(self, l1, l2):
        if(l1.lower() == l2.lower()): return True
        sim = self.similars.get(l1.lower(), '').lower()
        if(sim != ''): return sim == l2.lower()
        sim = self.similars.get(l2.lower(), '').lower()
        if(sim != ''): return sim == l1.lower()
        return False
    
    def words_same(self, w1, w2):
        return w1 ==w2

    def getdiff_word_html(self, v, w, xx: str, yh: str, showGreens: bool = False, showYellows: bool = False) -> str:
        if self.words_same(xx, yh):
            if not showGreens: return ''
            return f'<tr class="diff-row"><td class="common">{xx}</td><td class="common">{yh}</td></tr>'
        
        minlen = min(len(xx), len(yh))
        s1 = s2 = ''
        for i in range(minlen):
            if xx[i] == yh[i]:
                s1 += f'<span class="common">{xx[i]}</span>'
                s2 += f'<span class="common">{yh[i]}</span>'
                continue
            elif self.letters_similiar(xx[i], yh[i]):
                s1 += f'<span class="similar">{xx[i]}</span>'
                s2 += f'<span class="similar">{yh[i]}</span>'
                continue
            s1 += f'<span class="diff">{xx[i]}</span>'
            s2 += f'<span class="diff">{yh[i]}</span>'
        
        if len(xx) > minlen:
            s1 += f'<span class="diff">{xx[minlen:]}</span>'
        if len(yh) > minlen:
            s2 += f'<span class="diff">{yh[minlen:]}</span>'
        
        return f'<tr class="diff-row"><td>{s1}</td><td>{s2}</td></tr>'

    def generate_html(self, xml_sources, html_sources, book, chapter):
        self.html.append(f'<h2>{book.upper()} {chapter}</h2>')
        chapt = f"{book}_{chapter}"
        xm_c_len = len(xml_sources[chapt])
        hm_c_len = len(html_sources[chapt])
        minverses = min(xm_c_len, hm_c_len)
        
        self.html.append('<table class="diff-table">')
        
        for v in range(minverses):
            xm_v_len = len(xml_sources[chapt][v])
            hm_v_len = len(html_sources[chapt][v])
            minwords = min(xm_v_len, hm_v_len)
            
            self.html.append(f'<tr class="header"><td colspan="2">Verse {v+1}</td></tr>')
            
            for i in range(minwords):
                xx = xml_sources[chapt][v][i]["form"]
                yh = html_sources[chapt][v][i]["form"]
                self.html.append(self.getdiff_word_html(v+1, i+1, xx, yh, showGreens=True, showYellows=True))
            
            if xm_v_len != hm_v_len:
                if xm_v_len < hm_v_len:
                    strbuild = ' '.join(f"{html_sources[chapt][v][i]['form']}" for i in range(xm_v_len, hm_v_len))
                    self.html.append(f'<tr><td></td><td class="diff">+{strbuild}</td></tr>')
                else:
                    strbuild = ' '.join(f"{xml_sources[chapt][v][i]['form']}" for i in range(hm_v_len, xm_v_len))
                    self.html.append(f'<tr><td>-{strbuild}</td><td></td></tr>')
        
        self.html.append('</table></body></html>')
        return '\n'.join(self.html)
    
    


In [ ]:
GREEN = '<span class="green">'
RED = '<span class="red">'
YELLOW = '<span class="yellow">'
RESET = '</span>'
DIFF_RED = '<span class="diffred">'
DIFF_BLUE = '<span class="diffblue">'
class Visualizer_HTMLD(Visualizer_HTML):
    def __init__(self):
        self.similars = {
            "θ" : "Θ",
            "Θ" : "θ"
        }
        self.html = []
        self.html.append("""
        <!DOCTYPE html>
        <html>
        <head>
            <style>
                .diff-container { display: flex; flex-direction: column; gap: 1em; }
                .verse-row-header { display: flex; align-items: center; 
                background-color: #f0f0f0; }
                .common { background-color: #e6ffe6; }
                .diffx { background-color: #ffe6aa; }
                //.diffh { background-color: #ffe6e6; }
                .diffh {background-color: #ffe6aa;}
                .similar { background-color: #fff7e6; }
                .header { background-color: #f0f0f0; }
                .verse-row {
    display: flex;
    // flex-direction: column;
    align-items: flex-start;
    gap: 0.5em;
    min-height: 50px;
    padding: 0.25em;
}

.diffx {
    align-self: flex-start;
    background-color: #ffe6e6;
}

.diffh {
    align-self: flex-end;
    background-color: #fff7e6;//#e6ffe6;
}

.green {
    background-color: #e6ffe6;//#fff7e6;
    }

.yellow {
    background-color: #ffe6aa;
    }

.red {
    //background-color: #e6ffe6;/*#fff7e6;*/
    }    

.common {
    background-color: #e6ffe6;/*#fff7e6;*/
    align-self: anchor-center;
}

.diffred{
    background-color: #ff3333;
    align-self: anchor-center;
}
.diffblue{
    background-color: #3333ff;
    align-self: anchor-center;
}

/* Optional: responsive design */
@media (max-width: 768px) {
    .verse-row {
        gap: 0.25em;
        min-height: 40px;
    }
}
            </style>
        </head>
        <body>
        """)

    
    def letters_same(self, l1, l2):
        return self.words_same(l1, l2)
    
    def letters_similiar(self, l1, l2):
        if(l1.lower() == l2.lower()): return True
        sim = self.similars.get(l1.lower(), '').lower()
        if(sim!= ''): return sim == l2.lower()
        sim = self.similars.get(l2.lower(), '').lower()
        if(sim!= ''): return sim == l1.lower()
        return False
    
    def words_same(self, w1, w2):
        return w1 ==w2
    
    def getdiff_word_VisualHTM(self, v, w, xx: str, yh: str, showGreens: bool = False, showYellows: bool = False)-> str:
        p1 =""
        hadRed = False
        allReds = True
        COLOR = ''
        if(self.words_same(xx,yh)):
            allReds = False
            if not showGreens: return ''
            return self.w_colordiff_OK(v, w, xx, yh)
        else:
            
            minlen = min(len(xx), len(yh))
            s1=""
            s2=""
            for i in range(minlen):
                if(xx[i]==yh[i]):
                    allReds = False
                    s1+=f"{GREEN}{xx[i]}{RESET}"
                    s2+=f"{GREEN}{yh[i]}{RESET}"
                    continue
                else:
                    if(self.letters_similiar(xx[i], yh[i])):
                        allReds = False
                        s1+=f"{YELLOW}{xx[i]}{RESET}"
                        s2+=f"{YELLOW}{yh[i]}{RESET}"
                        continue
                    hadRed = True
                    s1+=f"{RED}{xx[i]}{RESET}"
                    s2+=f"{RED}{yh[i]}{RESET}"
            #print(f"minlen: {minlen}")
            if(len(xx)> minlen):
                s1+=f"{RED}{xx[minlen:]}{RESET}"
            if(len(yh)> minlen):
                s2+=f"{RED}{yh[minlen:]}{RESET}"
            if(True):
                if(hadRed):
                    COLOR = RED
                else:
                    if not showYellows: return ''
                    COLOR = YELLOW
        return f"{v}{s1}{RESET}\n{w}{s2}{RESET}"
    
    def w_colordiff_OK(self, v, w, xx: str, yh: str)-> str:
        COLOR = GREEN
        s1 = f"{xx}"
        s2 = f"{yh}"
        return f"{COLOR}{s1}\n{s2}{RESET}"


    def generate_html(self, xml_sources, html_sources, book, chapter):
        self.html.append(f'<h2>{book.upper()} {chapter}</h2>')
        chapt = f"{book}_{chapter}"
        xm_c_len = len(xml_sources[chapt])
        hm_c_len = len(html_sources[chapt])
        minverses = min(xm_c_len, hm_c_len)

        for v in range(minverses):
            xm_v_len = len(xml_sources[chapt][v])
            hm_v_len = len(html_sources[chapt][v])
            minwords = min(xm_v_len, hm_v_len)
            self.html.append(f'<div class="verse-row-header">Verse {v+1}</div>')
            
            verse_html = []
            for w in range (minwords):
                xx = xml_sources[chapt][v][w]["form"]
                yh = html_sources[chapt][v][w]["form"]
                if xx == yh:
                    verse_html.append(f'<span class="common">{xx}</span>')
                else:
                    verse_html.append(f'<span class="diff-container">')
                    verse_html.append(self.getdiff_word_VisualHTM('<span class="diffx">', '<span class="diffh">', xx, yh, showGreens=True, showYellows=True))
                    verse_html.append(f'</span>')
            seperator = " "
            if(xm_v_len < hm_v_len):
                strbuild =''
                for i in range(xm_v_len, hm_v_len):
                    #strbuild += f"{html_sources[chapt][v][i]['form']}{seperator}"
                    strbuild += f"{DIFF_RED}{html_sources[chapt][v][i]['form']}{seperator}{RESET}"
                #verse_html.append(f"{DIFF_RED}+{strbuild}{RESET}")
                verse_html.append(f'<span class="diffh">{DIFF_RED}+{RESET}</span>{strbuild}')
            else : 
                if (hm_v_len < xm_v_len):
                    strbuild =''
                    for i in range(hm_v_len, xm_v_len):
                        #strbuild += f"{xml_sources[ta(chapt)][v][i]['form']}{seperator}"
                        strbuild += f"{DIFF_BLUE}{xml_sources[ta(chapt)][v][i]['form']}{seperator}{RESET}"
                    #verse_html.append(f"{DIFF_BLUE}-{strbuild}{RESET}")
                    verse_html.append(f'<span class="diffx">{DIFF_BLUE}-{RESET}</span>{strbuild}')
            
            self.html.append(f'<div class="verse-row">{" ".join(verse_html)}</div>')
            
        if(xm_c_len> hm_c_len):
            for v in range(minverses, xm_c_len):
                xm_v_len = len(xml_sources[chapt][v])
                minwords = xm_v_len
                self.html.append(f'<div class="verse-row-header">++ Additional Verse {v+1}</div>')
                
                verse_html = []
                for w in range (minwords):
                    xx = xml_sources[chapt][v][w]["form"]
                    verse_html.append(f'<span class="common"><span class="diffh">{DIFF_RED}{xx}{RESET}</span>')
                
                self.html.append(f'<div class="verse-row">{" ".join(verse_html)}</div>')
                
        elif (xm_c_len< hm_c_len):
            for v in range(minverses, hm_c_len):
                hm_v_len = len(html_sources[chapt][v])
                minwords = hm_v_len
                self.html.append(f'<div class="verse-row-header">-- Additional Verse {v+1}</div>')
                
                verse_html = []
                for w in range (minwords):
                    yh = html_sources[chapt][v][w]["form"]
                    verse_html.append(f'<span class="common"><span class="diffh">{DIFF_RED}{yh}{RESET}</span>')
                
                self.html.append(f'<div class="verse-row">{" ".join(verse_html)}</div>')
        
        
        self.html.append('</body></html>')
        return '\n'.join(self.html)

In [ ]:
jd_only=True
bkslist = ["genesis", "exodus", "leviticus", "numbers", "deuteronomy", "joshua", "judges", "ruth", "1_samuel", "2_samuel", "1_kings", "2_kings", "1_chronicles", "2_chronicles", "ezra", "nehemiah", "esther", "job", "psalms", "proverbs", "ecclesiastes", "songs", "isaiah", "jeremiah", "lamentations", "ezekiel", "daniel", "hosea", "joel", "amos", "obadiah", "jonah", "micah", "nahum", "habakkuk", "zephaniah", "haggai", "zechariah", "malachi", "matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]
chps= [int(i) for i in "50 40 27 36 34 24 21 4 31 24 22 25 29 36 10 13 10 42 150 31 12 8 66 52 5 48 12 14 3 9 1 4 7 3 3 3 2 14 4 28 16 24 21 28 16 16 13 6 6 4 4 5 3 6 4 3 1 13 5 5 3 5 1 1 1 22".split(" ")]
idxl = {}
nt_stubchapters = ["matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]
for i in range (len(bkslist)):
    if (bkslist[i] in nt_stubchapters if jd_only else True):
        #idxl[bkslist[i].replace("_", "")[:3].upper()] = chps[i]
        idxl[bkslist[i] ] = chps[i]


In [ ]:
dw = Visualizer_HTMLD()
from datetime import datetime
from pathlib import Path
import os
def lostbook(b, c, h, x):
    chapt = f"{b}_{c}"
    if(chapt not in h.keys()): return 1
    if(chapt not in x.keys()): return -1
    return 0

date_str = datetime.now().strftime("%d-%m-%y")
dir_path = f"{date_str}-result-gk2"
Path(dir_path).mkdir(parents=True, exist_ok=True)
for current_book in nt_stubchapters:
    print(current_book)
    Path(dir_path+os.path.sep+current_book).mkdir(parents=True, exist_ok=True)
    for chapter in range(1, idxl[current_book]+1):
        dw = Visualizer_HTMLD()
        print(current_book+" "+str(chapter))
        with open(dir_path+os.path.sep+current_book+os.path.sep+str(chapter)+'.html', 'w', encoding='utf-8') as f:
            lb = lostbook(current_book, chapter, html_sources, xml_sources)
            if lb == 0:
                f.write(dw.generate_html(xml_sources, html_sources, current_book, chapter))
            else:
                f.write(f'<script type="text/javascript">window.alert("{current_book} {chapter} not found in xml source????!!?!?");</script>')
                f.write(dw.generate_html(html_sources, html_sources, current_book, chapter))

In [ ]:
print(nt_stubchapters[0])
current_book = nt_stubchapters[0]
print(html_sources.keys())
#len(html_sources[current_book])
html_sources["matthew_21b"] ## test failu neizdzesu ^^
xml_sources["matthew_21"]

# merge grieķi ar 65 24 

In [ ]:
GREEN = '<span class="green">'
RED = '<span class="red">'
YELLOW = '<span class="yellow">'
RESET = '</span>'
DIFF_RED = '<span class="diffred">'
DIFF_BLUE = '<span class="diffblue">'
class Visualizer_HTMLD4(Visualizer_HTML):
    def __init__(self):
        self.similars = {
            "θ" : "Θ",
            "Θ" : "θ"
        }
        self.html = []
        self.html.append("""
        <!DOCTYPE html>
        <html>
        <head>
            <style>
                .diff-container { display: flex; flex-direction: column; gap: 1em; }
                .verse-row-header { display: flex; align-items: center; 
                background-color: #f0f0f0; }
                .common { background-color: #e6ffe6; }
                .diffx { background-color: #ffe6aa; }
                /*.diffh { background-color: #ffe6e6; }*/
                .diffh {background-color: #ffe6aa;}
                .similar { background-color: #fff7e6; }
                .header { background-color: #f0f0f0; }
                .verse-row {
    display: flex;
    /* flex-direction: column;*/
    align-items: flex-start;
    gap: 0.5em;
    min-height: 50px;
    padding: 0.25em;
}

.diffx {
    align-self: flex-start;
    background-color: #ffe6e6;
}

.diffh {
    align-self: flex-end;
    background-color: #fff7e6;/*#e6ffe6;*/
}

.green {
    background-color: #e6ffe6;/*#fff7e6;*/
    }

.yellow {
    background-color: #ffe6aa;
    }

.red {
    /*background-color: #e6ffe6;*//*#fff7e6*/;
    }    

.common {
    background-color: #e6ffe6;/*#fff7e6;*/
    align-self: anchor-center;
}

.diffred{
    background-color: #ff3333;
    align-self: anchor-center;
}
.diffblue{
    background-color: #3333ff;
    align-self: anchor-center;
}

/* Optional: responsive design */
@media (max-width: 768px) {
    .verse-row {
        gap: 0.25em;
        min-height: 40px;
    }
}
            </style>
        </head>
        <body>
        """)

    
    def letters_same(self, l1, l2):
        return self.words_same(l1, l2)
    
    def letters_similiar(self, l1, l2):
        if(l1.lower() == l2.lower()): return True
        sim = self.similars.get(l1.lower(), '').lower()
        if(sim!= ''): return sim == l2.lower()
        sim = self.similars.get(l2.lower(), '').lower()
        if(sim!= ''): return sim == l1.lower()
        return False
    
    def words_same(self, w1, w2):
        return w1 ==w2
    
    def getdiff_word_VisualHTM(self, v, w, xx: str, yh: str, showGreens: bool = False, showYellows: bool = False)-> str:
        p1 =""
        hadRed = False
        allReds = True
        COLOR = ''
        if(self.words_same(xx,yh)):
            allReds = False
            if not showGreens: return ''
            return self.w_colordiff_OK(v, w, xx, yh)
        else:
            
            minlen = min(len(xx), len(yh))
            s1=""
            s2=""
            for i in range(minlen):
                if(xx[i]==yh[i]):
                    allReds = False
                    s1+=f"{GREEN}{xx[i]}{RESET}"
                    s2+=f"{GREEN}{yh[i]}{RESET}"
                    continue
                else:
                    if(self.letters_similiar(xx[i], yh[i])):
                        allReds = False
                        s1+=f"{YELLOW}{xx[i]}{RESET}"
                        s2+=f"{YELLOW}{yh[i]}{RESET}"
                        continue
                    hadRed = True
                    s1+=f"{RED}{xx[i]}{RESET}"
                    s2+=f"{RED}{yh[i]}{RESET}"
            #print(f"minlen: {minlen}")
            if(len(xx)> minlen):
                s1+=f"{RED}{xx[minlen:]}{RESET}"
            if(len(yh)> minlen):
                s2+=f"{RED}{yh[minlen:]}{RESET}"
            if(True):
                if(hadRed):
                    COLOR = RED
                else:
                    if not showYellows: return ''
                    COLOR = YELLOW
        return f"{v}{s1}{RESET}\n{w}{s2}{RESET}"
    
    def w_colordiff_OK(self, v, w, xx: str, yh: str)-> str:
        COLOR = GREEN
        s1 = f"{xx}"
        s2 = f"{yh}"
        return f"{COLOR}{s1}\n{s2}{RESET}"


    def generate_html(self, xml_sources, html_sources, g1_src, g2_src, book, booking, chapter):
        self.html.append(f'<h2>{book.upper()} {chapter}</h2>')
        chapt = f"{book}_{chapter}"
        chapting = f"{booking}_{chapter}"
        xm_c_len = len(xml_sources[chapt])
        hm_c_len = len(html_sources[chapt])
        minverses = min(xm_c_len, hm_c_len)

        for v in range(len(xml_sources[chapt])):
            #TODO : kāpēc jāņa 3. šeit nav??
            if(v<len(g1_src[chapting])):
                self.html.append(f'<div class="verse-row-header">Verse {v+1}</div>')
                
                xm_v_len = len(g1_src[chapting][v])
                hm_v_len = len(g2_src[chapting][v])
                minwords = min(xm_v_len, hm_v_len)
                verse_html = []
                for w in range (minwords):
                    xx = g1_src[chapting][v][w]["form"]
                    yh = g2_src[chapting][v][w]["form"]
                    if xx == yh:
                        verse_html.append(f'<span class="common">{xx}</span>')
                    else:
                        verse_html.append(f'<span class="diff-container">')
                        verse_html.append(self.getdiff_word_VisualHTM('<span class="diffx">', '<span class="diffh">', xx, yh, showGreens=True, showYellows=True))
                        verse_html.append(f'</span>')
                seperator = " "
                if(xm_v_len < hm_v_len):
                    strbuild =''
                    for i in range(xm_v_len, hm_v_len):
                        #strbuild += f"{g2_src[chapting][v][i]['form']}{seperator}"
                        strbuild += f"{DIFF_RED}{g2_src[chapting][v][i]['form']}{seperator}{RESET}"
                    #verse_html.append(f"{DIFF_RED}+{strbuild}{RESET}")
                    verse_html.append(f'<span class="diffh">{DIFF_RED}+{RESET}</span>{strbuild}')
                else : 
                    if (hm_v_len < xm_v_len):
                        strbuild =''
                        for i in range(hm_v_len, xm_v_len):
                            #strbuild += f"{g1_src[ta(chapting)][v][i]['form']}{seperator}"
                            strbuild += f"{DIFF_BLUE}{g1_src[ta(chapting)][v][i]['form']}{seperator}{RESET}"
                        #verse_html.append(f"{DIFF_BLUE}-{strbuild}{RESET}")
                        verse_html.append(f'<span class="diffx">{DIFF_BLUE}-{RESET}</span>{strbuild}')
                
                self.html.append(f'<div class="verse-row">{" ".join(verse_html)}</div>')
            
            #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~2465
            xm_v_len = len(xml_sources[chapt][v])
            hm_v_len = len(html_sources[chapt][v])
            minwords = min(xm_v_len, hm_v_len)
            verse_html = []
            for w in range (minwords):
                xx = xml_sources[chapt][v][w]["form"]
                yh = html_sources[chapt][v][w]["form"]
                if xx == yh:
                    verse_html.append(f'<span class="common">{xx}</span>')
                else:
                    verse_html.append(f'<span class="diff-container">')
                    verse_html.append(self.getdiff_word_VisualHTM('<span class="diffx">', '<span class="diffh">', xx, yh, showGreens=True, showYellows=True))
                    verse_html.append(f'</span>')
            seperator = " "
            if(xm_v_len < hm_v_len):
                strbuild =''
                for i in range(xm_v_len, hm_v_len):
                    #strbuild += f"{html_sources[chapt][v][i]['form']}{seperator}"
                    strbuild += f"{DIFF_RED}{html_sources[chapt][v][i]['form']}{seperator}{RESET}"
                #verse_html.append(f"{DIFF_RED}+{strbuild}{RESET}")
                verse_html.append(f'<span class="diffh">{DIFF_RED}+{RESET}</span>{strbuild}')
            else : 
                if (hm_v_len < xm_v_len):
                    strbuild =''
                    for i in range(hm_v_len, xm_v_len):
                        #strbuild += f"{xml_sources[ta(chapt)][v][i]['form']}{seperator}"
                        strbuild += f"{DIFF_BLUE}{xml_sources[ta(chapt)][v][i]['form']}{seperator}{RESET}"
                    #verse_html.append(f"{DIFF_BLUE}-{strbuild}{RESET}")
                    verse_html.append(f'<span class="diffx">{DIFF_BLUE}-{RESET}</span>{strbuild}')
            
            self.html.append(f'<div class="verse-row">{" ".join(verse_html)}</div>')
        
        self.html.append('</body></html>')
        return '\n'.join(self.html)

In [ ]:
#def biblecom_to_biblehub(s):
print(idxl)    

ajd_only=True
shrts = [ k for k in idxl.keys()]
abkslist = ["genesis", "exodus", "leviticus", "numbers", "deuteronomy", "joshua", "judges", "ruth", "1_samuel", "2_samuel", "1_kings", "2_kings", "1_chronicles", "2_chronicles", "ezra", "nehemiah", "esther", "job", "psalms", "proverbs", "ecclesiastes", "songs", "isaiah", "jeremiah", "lamentations", "ezekiel", "daniel", "hosea", "joel", "amos", "obadiah", "jonah", "micah", "nahum", "habakkuk", "zephaniah", "haggai", "zechariah", "malachi", "matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]
achps= [int(i) for i in "50 40 27 36 34 24 21 4 31 24 22 25 29 36 10 13 10 42 150 31 12 8 66 52 5 48 12 14 3 9 1 4 7 3 3 3 2 14 4 28 16 24 21 28 16 16 13 6 6 4 4 5 3 6 4 3 1 13 5 5 3 5 1 1 1 22".split(" ")]
bcom_to_bhub = {}
c = 0
ant_stubchapters = ["matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]
for i in range (len(abkslist)):
    if (abkslist[i] in ant_stubchapters if ajd_only else True):
        #idxl[bkslist[i].replace("_", "")[:3].upper()] = chps[i]
        bcom_to_bhub[shrts[c]] = abkslist[i]
        c+=1

In [ ]:
from datetime import datetime
from pathlib import Path
import os
SHOW_GREEK_DIFF = False

def lostbook(b, c, h, x):
    chapt = f"{b}_{c}"
    if(chapt not in h.keys()): return 1
    if(chapt not in x.keys()): return -1
    return 0
date_str = datetime.now().strftime("%d-%m-%y")
dir_path = f"{date_str}-result-merged"
Path(dir_path).mkdir(parents=True, exist_ok=True)
for current_book in idxl:
    print(names[current_book])
    Path(dir_path+os.path.sep+names[current_book]).mkdir(parents=True, exist_ok=True)
    for chapter in range(1, idxl[current_book]+1):
        dw = Visualizer_HTMLD4()
        print(names[current_book]+" "+str(chapter))
        with open(dir_path+os.path.sep+names[current_book]+os.path.sep+str(chapter)+'.html', 'w', encoding='utf-8') as f:
            lb = lostbook(current_book, chapter, html_sources, xml_sources)
            if lb == 0:
                f.write(dw.generate_html(result, result24, xml_sources if SHOW_GREEK_DIFF else html_sources, html_sources, current_book, bcom_to_bhub[current_book], chapter))
            else:
                #f.write(f'<script type="text/javascript">window.alert("{current_book} {chapter} not found in xml source????!!?!?");</script>')
                f.write(dw.generate_html(result, result24, html_sources, html_sources, current_book, bcom_to_bhub[current_book], chapter))

In [ ]:
# class TXTChunkRenderer():
    # def __init__(self):
        
    

In [ ]:
print(names)

In [ ]:
print(revnames)

In [ ]:
print(idxlIndex)

In [ ]:
print(html_sources.keys())
print(xml_sources.keys())

print(len(html_sources.keys()))
print(len(xml_sources.keys()))

# Find keys that are in html_sources but not in xml_sources
missing_keys = set(html_sources.keys()) - set(xml_sources.keys())

print("Keys in html_sources but missing in xml_sources:")
print(missing_keys)

In [ ]:
print(html_sources.keys())
print(result.keys())

print(len(html_sources.keys()))
print(len(result.keys()))

# Find keys that are in html_sources but not in result
missing_keys = set([bh_bc(i) for i in html_sources.keys()]) - set(result.keys())

print("Keys in html_sources but missing in result:")
print(missing_keys)

print ("~~~~~~~~~~~~~~~~~viceversa~~~~~~~~~~~~~~~~~~~~~~~")
missing_keys2 = set(result.keys()) - set([bh_bc(i) for i in html_sources.keys()])

print("Keys in result but missing in html_sources:")
print(missing_keys2)

# chunkit

In [ ]:
from datetime import datetime
from pathlib import Path
import os
CHUNK_SIZE_MAXCHARS="3000fff"
#CHUNK_SIZE_WORDS = 2000
def lostbook(b, c, h, x):
    chapt = f"{b}_{c}"
    if(chapt not in h.keys()): return 1
    if(chapt not in x.keys()): return -1
    return 0
date_str = datetime.now().strftime("%d-%m-%y")
dir_path = f"{date_str}-result-textedchunk"

bf=""
chunknum = 1
idxlIndex = [i for i in idxl.keys()]
#versex = 0
# pēc PANTUEM TIKAI :// TODO : teikumiem no xml
#wordex = 0  uuuuun xml a taču tieši trūkst kkādas daļas... nav sentence indexi :ROOROROROOROROR.....
# tāpēc vbnk mehāniski pēc chariem dalīts :(((((((( panti robežas.
semtences ={}


Path(dir_path).mkdir(parents=True, exist_ok=True)
for ibk in range(len(idxl)):
    current_book = idxlIndex[ibk]
    print(f'who is axios to open the biblios: {names[current_book]} ?')
    Path(dir_path+os.path.sep+names[current_book]).mkdir(parents=True, exist_ok=True)
    for chapter in range(1, idxl[current_book]+1):
        print(names[current_book]+" "+str(chapter))
        idxbhub = f'{names[current_book]}_{str(chapter)}'
        eoc = True
        
        while not eoc:
            for vnm in range(len(html_sources[idxbhub])):
                v = html_sources[idxbhub][vnm]
                vlv = result[idxbhub][vnm]
                tstr =" ".join([iii["form"] for iii in vlv]) +"\n"+ " ".join([iii["form"] for iii in v])
                if(len(bf) + len(tstr)>= CHUNK_SIZE_MAXCHARS):
                    print(bf)
                    eoc = True
                bf += tstr
                #versex+-1
                
            #with open(dir_path+os.path.sep+names[current_book]+os.path.sep+str(chapter)+str(chunknum)+'.txt', 'w', encoding='utf-8') as f:
            #    f.write(dw.generate_html(result, result24, html_sources, html_sources, current_book, bcom_to_bhub[current_book], chapter))

In [24]:
from datetime import datetime
from pathlib import Path
import os

CHUNK_SIZE_MAXCHARS = 1984 # jņ galilejas kāzās norāva beigas pēc 2500.... #3000

def lostbook(b, c, h, x):
    chapt = f"{b}_{c}"
    if chapt not in h.keys(): return 1
    if chapt not in x.keys(): return -1
    return 0

import unicodedata
def raccs_common(text):
    d = {
        #ord('\N{COMBINING ACUTE ACCENT}'):None,
        ord('’'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('‘'): None,  # LEFT SINGLE QUOTATION MARK
        ord('“'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('”'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,  # LEFT SINGLE QUOTATION MARK
        ord(']'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('-'): None,   # HYPHEN-MINUS
        ord('’'): None,   # HYPHEN-MINUS
        ord('⧼'): None,  # LEFT WHITE ANGLE BRACKET
        ord('⧽'): None,  # RIGHT WHITE ANGLE BRACKET
        ord('*'): None,   # ASTERISK
        ord('⇔'): None,  # LEFT RIGHT DOUBLE ARROW
        ord('〉'): None,  # GREATER-THAN SIGN
        ord('〈'): None,  # LESS-THAN SIGN
        ord('‿'): None,  # LOW LINE
        ord('«'): None,  # LEFT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('»'): None,  # RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('‹'): None,  # SINGLE LEFT-POINTING ANGLE QUOTATION MARK
        ord('›'): None,  # SINGLE RIGHT-POINTING ANGLE QUOTATION MARK
        ord('('): None,  # LEFT PARENTHESIS
        ord(')'): None,  # RIGHT PARENTHESIS
        ord('-') : None,  # HYPHEN-MINUS
        ord(';') : None,  # SEMICOLON
        }
    return unicodedata.normalize('NFD', text).translate(d)

def raccsGK(text):
    d = {
        ord('\N{COMBINING ACUTE ACCENT}'):None,
        0x0300: None,  # COMBINING GRAVE ACCENT
        0x0301: None,  # COMBINING ACUTE ACCENT
        0x0342: None,  # COMBINING GREEK PERISPOMENI
        0x0313: None,  # COMBINING COMMA ABOVE (PSILI)
        0x0314: None,  # COMBINING REVERSED COMMA ABOVE (DASIA)
        0x0308: None,  # COMBINING DIAERESIS
        0x0304: None,  # COMBINING MACRON
        0x0306: None,  # COMBINING BREVE
        0x0323: None,  # COMBINING DOT BELOW
        0x033E: None,  # COMBINING VERTICAL LINE ABOVE
        0x0345: None,  # COMBINING GREEK YPOGEGRAMMENI
        0x0360: None,  # COMBINING TILDE
        0x034F: None,  # COMBINING GRAPHEME JOINER
        0x035C: None,  # COMBINING DOUBLE BREVE
        0x035E: None,  # COMBINING DOUBLE MACRON
        0x037A: None,  # GREEK YPOGEGRAMMENI
        0x1FBD: None,  # GREEK KORONIS
        0x1FBF: None,  # GREEK PSILI
        0x1FBE: None,  # GREEK DASIA
        0x1FFF: None,   # GREEK OXIA
        ord('’'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('‘'): None,  # LEFT SINGLE QUOTATION MARK
        ord('“'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('”'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,  # LEFT SINGLE QUOTATION MARK
        ord(']'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('-'): None,   # HYPHEN-MINUS
        ord('’'): None,   # HYPHEN-MINUS
        ord('⧼'): None,  # LEFT WHITE ANGLE BRACKET
        ord('⧽'): None,  # RIGHT WHITE ANGLE BRACKET
        ord('*'): None,   # ASTERISK
        ord('⇔'): None,  # LEFT RIGHT DOUBLE ARROW
        ord('〉'): None,  # GREATER-THAN SIGN
        ord('〈'): None,  # LESS-THAN SIGN
        ord('‿'): None,  # LOW LINE
        ord('«'): None,  # LEFT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('»'): None,  # RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('‹'): None,  # SINGLE LEFT-POINTING ANGLE QUOTATION MARK
        ord('›'): None,  # SINGLE RIGHT-POINTING ANGLE QUOTATION MARK
        ord('('): None,  # LEFT PARENTHESIS
        ord(')'): None,  # RIGHT PARENTHESIS
# nestrādā .... nooooooooooooooooooooooo
        ord('-') : None,  # HYPHEN-MINUS
        ord(';') : None,  # SEMICOLON
        }
    return unicodedata.normalize('NFD', raccs_common(text)).translate(d)
    
def raccsLV(text):
    d = {
        }
    return unicodedata.normalize('NFD', raccs_common(text)).translate(d)

def create_chunked_output(idxl, html_sources, result, names):
    welc_text = "Κύριός Ιησους Κύριός Ἰησοῦς Χριστοσ\nΕἰρήνη τῷ οἴκῳ τούτῳ\n"
    date_str = datetime.now().strftime("%d-%m-%y")
    dir_path = f"{date_str}-result-textedchunk"
    Path(dir_path).mkdir(parents=True, exist_ok=True)
    
    idxlIndex = [i for i in idxl.keys()]
    for ibk in range(len(idxl)):
        current_book = idxlIndex[ibk]
        book_dir = Path(dir_path) / names[current_book]
        book_dir.mkdir(parents=True, exist_ok=True)

        buffer = raccsGK(welc_text)
        for chapter in range(1, idxl[current_book] + 1):
            print(f"{names[current_book]} {chapter}. nodaļa")
            
            chunk_num = 1
            verse_count = 0
            
            # Versiju apstrāde pa pāriem
            idxbhub = f'{names[current_book]}_{str(chapter)}'
            verses_in_chapter = len(html_sources[idxbhub])
            
            while verse_count < verses_in_chapter:
                # Iegūstam versijas no abām Bībelēm
                vlv = result[bh_bc(idxbhub)][verse_count] if verse_count < len(result[bh_bc(idxbhub)]) else None
                v = html_sources[idxbhub][verse_count] if verse_count < len(html_sources[idxbhub]) else None
                
                # Veidojam tekstu ar abām versijām
                if vlv and v:
                    if(verse_count == 0):
                        verse_text = raccsGK(welc_text) + raccsLV(" ".join([iii["form"] for iii in vlv])) + "\n" + \
                                    raccsGK(" ".join([iii["form"] for iii in v]))
                    else:                                
                        verse_text = raccsLV(" ".join([iii["form"] for iii in vlv])) + "\n" + \
                                raccsGK(" ".join([iii["form"] for iii in v])+ "\n")
                elif vlv:
                    verse_text = raccsLV(welc_text + " ".join([iii["form"] for iii in vlv]))+ "\n"
                elif v:
                    verse_text = raccsGK(welc_text + raccsGK(" ".join([iii["form"] for iii in v]))+ "\n")
                else:
                    verse_text = ""
                                
                if(len(welc_text+ verse_text)>= CHUNK_SIZE_MAXCHARS):
                    print(f"help, big verse at {names[current_book]}_{str(chapter)} : {verse_count}")
                    # Saglabājam pilno gabalu
                    buffer+= raccsGK(welc_text)+ verse_text
                    chunk_path = book_dir / f"{chapter}_{chunk_num}.txt"
                    with open(chunk_path, 'w', encoding='utf-8') as f:
                        f.write(buffer.strip())
                    
                    # Sākam jaunu gabalu ar pašreizējo versi
                    buffer = raccsGK(welc_text)+ verse_text
                    chunk_num += 1
                
                # Pārbaudam vai var pievienot tekstu pašreizējam gabalam
                # +1 jo newline.
                if len(buffer) +1 + len(verse_text) >= CHUNK_SIZE_MAXCHARS:
                    # Saglabājam pilno gabalu
                    chunk_path = book_dir / f"{chapter}_{chunk_num}.txt"
                    with open(chunk_path, 'w', encoding='utf-8') as f:
                        f.write(buffer.strip())
                    
                    # Sākam jaunu gabalu ar pašreizējo versi
                    buffer = raccsGK(welc_text)+ verse_text
                    chunk_num += 1
                else:
                    # Pievienojam tekstu esošajam gabalam
                    buffer += "\n" + verse_text
                verse_text=""
                verse_count += 1
            
            # Saglabājam pēdējo gabalu, ja tas nav tukšs
            if len(buffer.strip()) >= len(welc_text):
                chunk_path = book_dir / f"{chapter}_{chunk_num}.txt"
                with open(chunk_path, 'w', encoding='utf-8') as f:
                    f.write(buffer.strip())
                buffer=""

# Lietošanas piemērs:
create_chunked_output(idxl, html_sources, result, names)
!cd ~/05-04-25-result-textedchunk && zip 04-05-chun-65-NU.zip . -r && cd ~

matthew 1. nodaļa
matthew 2. nodaļa
matthew 3. nodaļa
matthew 4. nodaļa
matthew 5. nodaļa
matthew 6. nodaļa
matthew 7. nodaļa
matthew 8. nodaļa
matthew 9. nodaļa
matthew 10. nodaļa
matthew 11. nodaļa
matthew 12. nodaļa
matthew 13. nodaļa
matthew 14. nodaļa
matthew 15. nodaļa
matthew 16. nodaļa
matthew 17. nodaļa
matthew 18. nodaļa
matthew 19. nodaļa
matthew 20. nodaļa
matthew 21. nodaļa
matthew 22. nodaļa
matthew 23. nodaļa
matthew 24. nodaļa
matthew 25. nodaļa
matthew 26. nodaļa
matthew 27. nodaļa
matthew 28. nodaļa
mark 1. nodaļa
mark 2. nodaļa
mark 3. nodaļa
mark 4. nodaļa
mark 5. nodaļa
mark 6. nodaļa
mark 7. nodaļa
mark 8. nodaļa
mark 9. nodaļa
mark 10. nodaļa
mark 11. nodaļa
mark 12. nodaļa
mark 13. nodaļa
mark 14. nodaļa
mark 15. nodaļa
mark 16. nodaļa
luke 1. nodaļa
luke 2. nodaļa
luke 3. nodaļa
luke 4. nodaļa
luke 5. nodaļa
luke 6. nodaļa
luke 7. nodaļa
luke 8. nodaļa
luke 9. nodaļa
luke 10. nodaļa
luke 11. nodaļa
luke 12. nodaļa
luke 13. nodaļa
luke 14. nodaļa
luke 15. nodaļa

john 5. nodaļa
john 6. nodaļa
john 7. nodaļa
john 8. nodaļa
john 9. nodaļa
john 10. nodaļa
john 11. nodaļa
john 12. nodaļa
john 13. nodaļa
john 14. nodaļa
john 15. nodaļa
john 16. nodaļa
john 17. nodaļa
john 18. nodaļa
john 19. nodaļa
john 20. nodaļa
john 21. nodaļa
acts 1. nodaļa
acts 2. nodaļa
acts 3. nodaļa
acts 4. nodaļa
acts 5. nodaļa
acts 6. nodaļa
acts 7. nodaļa
acts 8. nodaļa
acts 9. nodaļa
acts 10. nodaļa
acts 11. nodaļa
acts 12. nodaļa
acts 13. nodaļa
acts 14. nodaļa
acts 15. nodaļa
acts 16. nodaļa
acts 17. nodaļa
acts 18. nodaļa
acts 19. nodaļa
acts 20. nodaļa
acts 21. nodaļa
acts 22. nodaļa
acts 23. nodaļa
acts 24. nodaļa
acts 25. nodaļa
acts 26. nodaļa
acts 27. nodaļa
acts 28. nodaļa
romans 1. nodaļa
romans 2. nodaļa
romans 3. nodaļa
romans 4. nodaļa
romans 5. nodaļa
romans 6. nodaļa
romans 7. nodaļa
romans 8. nodaļa
romans 9. nodaļa
romans 10. nodaļa
romans 11. nodaļa
romans 12. nodaļa
romans 13. nodaļa
romans 14. nodaļa
romans 15. nodaļa
romans 16. nodaļa
1_corinthians 1

In [ ]:
# for i in bcom_to_bhub:
#     print(bcom_to_bhub[i])
#     print(i)
# result['MAT_28']

In [ ]:
#xml_sources[f'{bcom_to_bhub[[i for i in idxl.keys()][0]]}_3']
